In [ ]:
import itertools as it
import numpy as np
import pandas as pd
import pingouin as pg
from scipy.stats import median_abs_deviation as mad

pop = pd.read_csv("C:/Uner/Semester 5/Multivariat/Coolyeah/pop_sample.csv")
rock = pd.read_csv("C:/Uner/Semester 5/Multivariat/Coolyeah/rock_sample.csv")
df = pd.concat([pop, rock])
df['group'] = ['pop'] * len(pop) + ['rock'] * len(rock)
df['group'] = df['group'].map({'pop': 0, 'rock': 1})
df['group'] = df['group'].astype('category')
df = df.drop(columns=['X','track_id'])
from scipy.stats import levene

def homogeneity_levene(X: np.ndarray, groups: np.ndarray, center: str = "mean"):
    gvals = pd.unique(pd.Series(groups))
    k = len(gvals)
    if k < 2:
        raise ValueError("Levene needs at least 2 groups.")
    N, p = X.shape
    F_list, P_list = [], []
    for j in range(p):
        samples = [X[groups == g, j] for g in gvals]
        stat, pval = levene(*samples, center=center)
        F_list.append(float(stat))
        P_list.append(float(pval))
    return {"F": F_list, "pvals": P_list, "p_min": float(np.min(P_list)),
            "df1": int(k - 1), "df2": int(N - k), "k": k, "p": p}

def _mad_trim_df(df, cols, k=3.0):
    """Per-group MAD trimming on the selected columns."""
    g = df.copy()
    for c in cols:
        med = g[c].median()
        m = mad(g[c], scale="normal", nan_policy="omit")
        if m and not np.isnan(m):
            z = (g[c] - med).abs() / m
            g = g[z <= k]
    return g

def _to_numeric(df, cols):
    """Coerce listed cols to numeric, NaNs on failure; drop rows with any NaN in cols."""
    out = df.copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    # replace infs then drop NaNs
    out[cols] = out[cols].replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=cols)
    return out

def scan_all_triplets(
    df: pd.DataFrame,
    group_col: str = "group",
    alpha: float = 0.05,
    trim_k: float = 3.0,
    standardize: bool = True,
    min_total_n: int = 100,
    min_per_group: int = 30,
):
    """
    Scan all 3-feature combos in df (numeric columns only) and run pingouin.multivariate_ttest
    robustly (dtype-safe). Returns a results DataFrame sorted by p-value.
    """
    # numeric candidate columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # also allow numeric-looking objects (coerce check)
    maybe_obj = [c for c in df.columns if c not in numeric_cols and c != group_col]
    for c in maybe_obj:
        # quick sniff: if >80% values parse to number, keep as candidate
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().mean() > 0.8:
            numeric_cols.append(c)

    numeric_cols = [c for c in numeric_cols if c != group_col]
    numeric_cols = list(dict.fromkeys(numeric_cols))  # de-dupe, keep order

    if len(numeric_cols) < 3:
        raise ValueError("Need at least 3 numeric columns to scan triplets.")

    # enforce group to string labels
    d0 = df.copy()
    d0[group_col] = d0[group_col].astype(str)

    # gather groups and basic size check
    groups = sorted(d0[group_col].unique())
    if len(groups) != 2:
        raise ValueError(f"`{group_col}` must have exactly 2 groups; found: {groups}")
    g1, g2 = groups

    rows = []
    for feats in it.combinations(numeric_cols, 3):
        # coerce to numeric and clean
        d = _to_numeric(d0[[group_col, *feats]].copy(), list(feats))

        # optional standardize (before trimming to make MAD comparable)
        if standardize:
            d[list(feats)] = (d[list(feats)] - d[list(feats)].mean()) / d[list(feats)].std(ddof=1)

        # per-group MAD trim
        d_g1 = _mad_trim_df(d[d[group_col] == g1], list(feats), k=trim_k)
        d_g2 = _mad_trim_df(d[d[group_col] == g2], list(feats), k=trim_k)
        d_clean = pd.concat([d_g1, d_g2], axis=0, ignore_index=True)

        n1 = (d_clean[group_col] == g1).sum()
        n2 = (d_clean[group_col] == g2).sum()
        n_tot = n1 + n2

        # minimal size guards
        if n_tot < min_total_n or n1 < min_per_group or n2 < min_per_group:
            rows.append({
                "features": feats, "n1": n1, "n2": n2, "n": n_tot,
                "T2": np.nan, "F": np.nan, "df1": np.nan, "df2": np.nan, "p": np.nan,
                "note": "skipped: too few samples after cleaning"
            })
            continue

        # prepare pure float arrays for pingouin
        X1 = d_clean.loc[d_clean[group_col] == g1, feats].to_numpy(dtype=float)
        X2 = d_clean.loc[d_clean[group_col] == g2, feats].to_numpy(dtype=float)

        # pingouin drops rows with NaN internally, but we already cleaned
        try:
            ht2 = pg.multivariate_ttest(X1, X2)  # independent two-sample
            # parse result across PG versions
            # expected columns: ['T2', 'F', 'df1', 'df2', 'p-val']
            # pingouin returns a DataFrame with one row
            if isinstance(ht2, pd.DataFrame):
                T2   = float(ht2.get("T2",   pd.Series([np.nan])).iloc[0])
                F    = float(ht2.get("F",    pd.Series([np.nan])).iloc[0])
                df1  = int(ht2.get("df1",    pd.Series([np.nan])).iloc[0])
                df2  = int(ht2.get("df2",    pd.Series([np.nan])).iloc[0])
                pval = float(ht2.get("p-val",pd.Series([np.nan])).iloc[0])
            else:
                T2 = F = df1 = df2 = pval = np.nan
            rows.append({
                "features": feats, "n1": n1, "n2": n2, "n": n_tot,
                "T2": T2, "F": F, "df1": df1, "df2": df2, "p": pval, "note": ""
            })
        except Exception as e:
            rows.append({
                "features": feats, "n1": n1, "n2": n2, "n": n_tot,
                "T2": np.nan, "F": np.nan, "df1": np.nan, "df2": np.nan, "p": np.nan,
                "note": f"error: {type(e).__name__}: {e}"
            })

    res = pd.DataFrame(rows)
    # sort with valid p-values first, then by p asc
    res["_valid"] = res["p"].notna()
    res = res.sort_values(by=["_valid", "p"], ascending=[False, True]).drop(columns="_valid").reset_index(drop=True)
    return res

# Make sure df has a binary 'group' column (e.g., 'ckd' vs 'notckd', 'stroke' vs 'no_stroke', etc.)
# and your numeric columns (possibly mixed types) are present.

out = scan_all_triplets(df, group_col="group",
                        alpha=0.05, trim_k=3.0,
                        standardize=True,
                        min_total_n=100, min_per_group=30)
out